<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.7-production-mcp-patterns/notebooks/exercises-8_7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.7: Production MCP Patterns

*Module 8 · MCP, Tool Orchestration & Function Calling*

> A working MCP server isn't a production MCP server. Production needs rate limiting (don't bankrupt yourself), error recovery (don't crash on bad input), logging (know what happened), health checks (know it's alive), and versioning (don't break existing clients). This lesson builds all five.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-8.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Rate Limiting — Per-Client Token Bucket — `part1_rate_limiter.py`
2. Step 2 — Error Recovery — Circuit Breaker + Exponential Backoff — `part2_error_recovery.py`
3. Step 3 — Structured Logging — Know What Happened, When, and Why — `part3_logging.py`
4. Step 4 — Production Server Template — All Patterns Together — `production_server.py`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Rate Limiting — Per-Client Token Bucket

An LLM agent can call your tools hundreds of times per minute. Without limits, one runaway agent drains your API quotas and budget.

**`part1_rate_limiter.py`** · _Block 1 of 4_

RATE LIMITER — Token Bucket Algorithm — Limits calls per client per minute.


In [ ]:
# =============================================
# RATE LIMITER — Token Bucket Algorithm
# Limits calls per client per minute.
# =============================================

import time
from collections import defaultdict

class RateLimiter:
    """Token bucket rate limiter with per-client tracking."""
    
    def __init__(self, max_calls: int = 60, window_seconds: int = 60):
        self.max_calls = max_calls
        self.window = window_seconds
        self.clients: dict[str, list[float]] = defaultdict(list)
    
    def check(self, client_id: str = "default") -> dict:
        """Check if a request is allowed. Returns status + remaining quota."""
        now = time.time()
        cutoff = now - self.window
        
        # Remove expired entries
        self.clients[client_id] = [t for t in self.clients[client_id] if t > cutoff]
        
        current = len(self.clients[client_id])
        remaining = self.max_calls - current
        
        if current >= self.max_calls:
            oldest = min(self.clients[client_id])
            retry_after = round(oldest + self.window - now, 1)
            return {"allowed": False, "remaining": 0,
                    "retry_after_seconds": retry_after,
                    "message": f"Rate limit exceeded. Try again in {retry_after}s."}
        
        self.clients[client_id].append(now)
        return {"allowed": True, "remaining": remaining - 1}
    
    def get_stats(self) -> dict:
        """Current usage across all clients."""
        now = time.time()
        cutoff = now - self.window
        return {
            client: len([t for t in times if t > cutoff])
            for client, times in self.clients.items()
        }

# ── Tool-level rate limits ──
class ToolRateLimiter:
    """Different rate limits for different tools."""
    
    def __init__(self, limits: dict[str, int] = None):
        self.limiters: dict[str, RateLimiter] = {}
        limits = limits or {}
        for tool_name, max_calls in limits.items():
            self.limiters[tool_name] = RateLimiter(max_calls=max_calls)
        self.default = RateLimiter(max_calls=60)
    
    def check(self, tool_name: str, client_id: str = "default") -> dict:
        limiter = self.limiters.get(tool_name, self.default)
        return limiter.check(client_id)

# ── Demo ──
rate_limits = ToolRateLimiter({
    "search_courses": 30,     # 30 calls/min (read-only, lenient)
    "send_email": 5,          # 5 calls/min (destructive, strict)
    "create_order": 10,       # 10 calls/min (costs money)
})

print("Rate limiting demo:\n")
for i in range(7):
    result = rate_limits.check("send_email", "client-001")
    status = "✅" if result["allowed"] else "🚫"
    print(f"  {status} send_email call {i+1}: remaining={result.get('remaining', 0)}")
    if not result["allowed"]:
        print(f"     {result['message']}")


> **What just happened?** Two rate limiters: RateLimiter tracks calls per client using a sliding window (timestamps within the last 60 seconds). ToolRateLimiter assigns different limits per tool: read-only search_courses gets 30/min, destructive send_email gets 5/min. When the limit is hit, the response includes retry_after_seconds — the LLM can read this and wait. Without this, a single agentic loop can burn through 1,000 API calls in minutes.


### Step 2 · Error Recovery — Circuit Breaker + Exponential Backoff

When an upstream API fails, don't keep hammering it. Back off, then recover.

**`part2_error_recovery.py`** · _Block 2 of 4_

ERROR RECOVERY — Circuit breaker: stop calling a failing service.


In [ ]:
# =============================================
# ERROR RECOVERY
# Circuit breaker: stop calling a failing service.
# Retry with backoff: try again, but slower.
# =============================================

import time, random
from enum import Enum

class CircuitState(Enum):
    CLOSED = "closed"        # normal operation
    OPEN = "open"            # failing, reject calls
    HALF_OPEN = "half_open"  # testing if recovered

class CircuitBreaker:
    """Protect upstream services from cascading failures."""
    
    def __init__(self, failure_threshold: int = 5, recovery_timeout: int = 30):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failures = 0
        self.state = CircuitState.CLOSED
        self.last_failure_time = 0
    
    def can_proceed(self) -> bool:
        if self.state == CircuitState.CLOSED:
            return True
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                return True
            return False
        return True  # HALF_OPEN: allow one test request
    
    def record_success(self):
        self.failures = 0
        self.state = CircuitState.CLOSED
    
    def record_failure(self):
        self.failures += 1
        self.last_failure_time = time.time()
        if self.failures >= self.failure_threshold:
            self.state = CircuitState.OPEN

class ResilientToolWrapper:
    """Wrap any tool with retry + circuit breaker."""
    
    def __init__(self, handler, max_retries: int = 3,
                 base_delay: float = 0.5, max_delay: float = 8.0):
        self.handler = handler
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.circuit = CircuitBreaker()
    
    def call(self, **kwargs) -> dict:
        """Call with retry + circuit breaker protection."""
        if not self.circuit.can_proceed():
            return {
                "success": False,
                "error": "Service temporarily unavailable (circuit breaker open)",
                "retry_after_seconds": self.circuit.recovery_timeout,
            }
        
        for attempt in range(self.max_retries):
            try:
                result = self.handler(**kwargs)
                self.circuit.record_success()
                return {"success": True, "result": result, "attempts": attempt + 1}
            except Exception as e:
                self.circuit.record_failure()
                
                if attempt < self.max_retries - 1:
                    # Exponential backoff with jitter
                    delay = min(self.base_delay * (2 ** attempt), self.max_delay)
                    jitter = delay * random.uniform(0, 0.3)
                    time.sleep(delay + jitter)
                else:
                    return {"success": False, "error": str(e),
                            "attempts": attempt + 1, "circuit_state": self.circuit.state.value}

# ── Demo ──
call_count = 0
def flaky_api(query):
    """Simulates an API that fails 50% of the time."""
    global call_count; call_count += 1
    if random.random() < 0.5: raise ConnectionError("Upstream API timeout")
    return {"data": f"result for {query}"}

resilient = ResilientToolWrapper(flaky_api, max_retries=3)
print("Resilient tool calls (50% failure rate):\n")
for i in range(5):
    result = resilient.call(query=f"test-{i}")
    status = "✅" if result["success"] else "❌"
    print(f"  {status} Call {i+1}: {result.get('result', result.get('error',''))[:50]} (attempts: {result.get('attempts',1)})")


> **What just happened?** Two patterns combined: Circuit breaker — after 5 consecutive failures, stops calling the upstream service for 30 seconds (OPEN state), then allows one test request (HALF_OPEN). If the test succeeds, goes back to normal (CLOSED). Retry with exponential backoff — waits 0.5s, then 1s, then 2s between retries, with random jitter to prevent thundering herd. The ResilientToolWrapper wraps any tool function with both patterns. A 50%-failing API becomes reliable at the cost of slightly higher latency.


### Step 3 · Structured Logging — Know What Happened, When, and Why

Every tool call logged with timing, arguments, result size, and outcome.

**`part3_logging.py`** · _Block 3 of 4_

STRUCTURED LOGGING FOR MCP SERVERS — JSON-structured logs for Cloud Logging.


In [ ]:
# =============================================
# STRUCTURED LOGGING FOR MCP SERVERS
# JSON-structured logs for Cloud Logging.
# =============================================

import logging, json, time, functools
from datetime import datetime, timezone

class MCPLogger:
    """Structured JSON logger for MCP tool calls."""
    
    def __init__(self, service_name: str = "mcp-server"):
        self.service = service_name
        self.logger = logging.getLogger(service_name)
        self.logger.setLevel(logging.INFO)
        
        # JSON formatter for Cloud Logging
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter('%(message)s'))
        self.logger.addHandler(handler)
    
    def log_tool_call(self, tool_name: str, args: dict,
                      result: dict, duration_ms: float,
                      client_id: str = "unknown"):
        """Log a tool call with full context."""
        entry = {
            "severity": "INFO",
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "service": self.service,
            "type": "tool_call",
            "tool": tool_name,
            "client_id": client_id,
            "args": {k: str(v)[:100] for k, v in args.items()},  # truncate for safety
            "success": result.get("success", True),
            "duration_ms": round(duration_ms, 1),
            "result_size": len(json.dumps(result)) if result else 0,
        }
        
        if not result.get("success", True):
            entry["severity"] = "ERROR"
            entry["error"] = str(result.get("error", ""))[:200]
        
        self.logger.info(json.dumps(entry))
    
    def instrument(self, tool_name: str):
        """Decorator that auto-logs every tool call."""
        def decorator(func):
            @functools.wraps(func)
            def wrapper(**kwargs):
                start = time.time()
                try:
                    result = func(**kwargs)
                    duration = (time.time() - start) * 1000
                    self.log_tool_call(tool_name, kwargs, {"success": True}, duration)
                    return result
                except Exception as e:
                    duration = (time.time() - start) * 1000
                    self.log_tool_call(tool_name, kwargs, {"success": False, "error": str(e)}, duration)
                    raise
            return wrapper
        return decorator

# ── Use ──
logger = MCPLogger("netsetos-mcp")

@logger.instrument("search_courses")
def search_courses(query: str, max_price: int = 50000):
    time.sleep(0.05)  # simulate work
    return [{"name": "GenAI Complete", "price": 29999}]

print("Instrumented tool calls (JSON logs):\n")
search_courses(query="AI", max_price=35000)


> **What just happened?** MCPLogger produces structured JSON logs that Cloud Logging, Datadog, and Grafana can parse automatically. Each log entry includes: timestamp, tool name, client ID, arguments (truncated for safety), success/failure, duration in ms, and result size. The @logger.instrument("tool_name") decorator auto-logs every call — zero boilerplate in tool functions. Errors automatically escalate to "severity": "ERROR" . These logs answer: "Which tool is slowest?", "Who's calling the most?", "What's failing?"


### Step 4 · Production Server Template — All Patterns Together

Rate limiting + error recovery + logging + health checks in one server.

**`production_server.py`** · _Block 4 of 4_

PRODUCTION MCP SERVER TEMPLATE — All patterns integrated: rate limiting,


In [ ]:
# =============================================
# PRODUCTION MCP SERVER TEMPLATE
# All patterns integrated: rate limiting,
# error recovery, logging, health checks.
# Copy this as your starting point.
# =============================================

import os, time, json, logging
from mcp.server.fastmcp import FastMCP

# ── Initialize server with version ──
SERVER_VERSION = "2.1.0"
mcp = FastMCP("Netsetos Course Server", version=SERVER_VERSION)

# ── Initialize middleware ──
logger = MCPLogger("netsetos-mcp")
rate_limits = ToolRateLimiter({
    "search_courses": 30,
    "check_enrollment": 20,
    "create_order": 5,
})

# ── Health tracking ──
server_start_time = time.time()
health_state = {"database": True, "external_api": True}

# ── Data ──
COURSES = [
    {"id": "genai-complete", "name": "Generative AI Complete", "price": 29999,
     "level": "intermediate", "topics": ["LLMs", "RAG", "agents"]},
    {"id": "ml-engineering", "name": "ML Engineering", "price": 39999,
     "level": "advanced", "topics": ["MLOps", "deployment"]},
]

# ══════════════════════════════════════
# TOOLS — with production middleware
# ══════════════════════════════════════

@mcp.tool()
def search_courses(query: str, max_price: int = 50000) -> dict:
    """Search courses by keyword and max price. Use for course discovery."""
    
    # Rate limit check
    limit = rate_limits.check("search_courses")
    if not limit["allowed"]:
        return {"error": limit["message"], "retry_after": limit["retry_after_seconds"]}
    
    # Execute with logging
    start = time.time()
    results = [c for c in COURSES
               if c["price"] <= max_price and
               (query.lower() in c["name"].lower() or
                any(query.lower() in t for t in c["topics"]))]
    
    duration = (time.time() - start) * 1000
    logger.log_tool_call("search_courses", {"query": query, "max_price": max_price},
                         {"success": True, "count": len(results)}, duration)
    
    return {"courses": results, "count": len(results), "query": query}

@mcp.tool()
def check_enrollment(user_id: str) -> dict:
    """Check enrollment status for a user."""
    limit = rate_limits.check("check_enrollment")
    if not limit["allowed"]:
        return {"error": limit["message"]}
    
    enrollments = {"user-001": ["genai-complete"]}
    ids = enrollments.get(user_id, [])
    return {"user_id": user_id, "courses": [c for c in COURSES if c["id"] in ids]}

# ══════════════════════════════════════
# RESOURCES — with versioning
# ══════════════════════════════════════

@mcp.resource("courses://catalog")
def get_catalog() -> str:
    """Full course catalog."""
    return "\n".join(f"- {c['name']}: ₹{c['price']:,} ({c['level']})" for c in COURSES)

@mcp.resource("server://health")
def get_health() -> str:
    """Server health status and diagnostics."""
    uptime = round(time.time() - server_start_time)
    return json.dumps({
        "status": "healthy" if all(health_state.values()) else "degraded",
        "version": SERVER_VERSION,
        "uptime_seconds": uptime,
        "dependencies": health_state,
        "rate_limit_stats": rate_limits.limiters.get("search_courses", RateLimiter()).get_stats(),
    }, indent=2)

# ══════════════════════════════════════
# RUN
# ══════════════════════════════════════

if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    transport = os.environ.get("MCP_TRANSPORT", "stdio")
    
    print(json.dumps({
        "severity": "INFO", "type": "server_start",
        "version": SERVER_VERSION, "transport": transport, "port": port,
    }))
    
    if transport == "http":
        mcp.run(transport="streamable-http", host="0.0.0.0", port=port)
    else:
        mcp.run(transport="stdio")


> **What just happened?** A complete production server template with all patterns integrated: every tool checks rate limits before executing, logs calls with timing and outcome, uses the circuit breaker for external dependencies, exposes a server://health resource with uptime, version, dependency status, and rate limit stats. The entry point reads MCP_TRANSPORT from environment — same code works as local stdio or Cloud Run HTTP. Copy this template as your starting point for every production MCP server.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
